# Phase 2 - Exploratory Data Analysis
**IndiaKart E-Commerce Analytics | Analyst: Bhavana**

Ten charts with written observations. Every chart is also saved as a PNG in `../outputs/charts/`.

In [1]:

import pandas as pd, numpy as np, json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.titlesize"] = 13

DATA = "../data"
CHARTS = "../outputs/charts"
TABLES = "../outputs/tables"
os.makedirs(CHARTS, exist_ok=True); os.makedirs(TABLES, exist_ok=True)

def rd(name, dates=()):
    df = pd.read_csv(f"{DATA}/{name}.csv")
    for c in dates:
        df[c] = pd.to_datetime(df[c], format="%d-%m-%Y", errors="coerce")
    return df

def inr(v):
    return f"Rs.{v:,.0f}"

orders     = rd("orders", ["order_date", "delivered_date"])
order_items= rd("order_items")
customers  = rd("customers", ["registration_date", "last_login_date"])
products   = rd("products", ["launch_date"])
payments   = rd("payments", ["payment_date", "refund_date"])
returns    = rd("returns", ["return_date", "refund_date"])
inventory  = rd("inventory", ["last_restocked_date"])
suppliers  = rd("suppliers", ["created_date"])
print("All 8 tables loaded")


All 8 tables loaded


In [2]:
orders["order_month"] = orders["order_date"].dt.to_period("M")
orders["year"] = orders["order_date"].dt.year
oc = orders.merge(customers[["customer_id", "segment", "age", "gender"]], on="customer_id", how="left")
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"{CHARTS}/{name}.png", dpi=140, bbox_inches="tight"); plt.show()
print("ready")

ready


## Chart 1 - Monthly order volume (Diwali months highlighted)

In [3]:
m = orders.groupby("order_month").size()
x = m.index.astype(str)
colors = ["#d94f04" if str(p)[-2:] in ("10", "11") else "#2a6f97" for p in m.index]
fig, ax = plt.subplots()
ax.bar(x, m.values, color=colors)
ax.set_title("Monthly order volume (orange = Diwali months, Oct/Nov)")
ax.set_xlabel("Month"); ax.set_ylabel("Orders")
plt.xticks(rotation=90)
save(fig, "01_monthly_order_volume")
m.to_csv(f"{TABLES}/monthly_orders.csv")

**Observation:** Order volume is broadly stable month to month, with visible lifts in the
October-November festive window in both years. The Diwali peaks confirm that inventory and
logistics capacity should be planned around Q3 of the financial year.

## Chart 2 - Monthly revenue (GMV) with year-on-year comparison

In [4]:
gmv = orders.groupby("order_month")["final_amount"].sum()/1e7
fig, ax = plt.subplots()
ax.plot(gmv.index.astype(str), gmv.values, marker="o", color="#1b4965")
ax.set_title("Monthly GMV (Rs. crore)"); ax.set_ylabel("GMV (Rs. cr)"); ax.set_xlabel("Month")
plt.xticks(rotation=90)
save(fig, "02_monthly_gmv")

yoy = orders.groupby([orders["order_date"].dt.year, orders["order_date"].dt.month])["final_amount"].sum().unstack(0)/1e7
fig, ax = plt.subplots()
yoy.plot(ax=ax, marker="o")
ax.set_title("GMV by calendar month, year over year (Rs. crore)")
ax.set_xlabel("Month number"); ax.set_ylabel("GMV (Rs. cr)")
save(fig, "02b_gmv_yoy")
gmv.to_csv(f"{TABLES}/monthly_gmv.csv")

**Observation:** GMV tracks order volume closely, so growth is coming from more orders rather
than from higher basket values. The year-on-year view shows 2024 and 2025 running at a similar
level per month, with the festive quarter the clear high point.

## Chart 3 - Category-wise revenue share

In [5]:
cat = order_items.groupby("category")["total_price"].sum().sort_values()/1e7
fig, ax = plt.subplots()
ax.barh(cat.index, cat.values, color="#2a9d8f")
ax.set_title("Revenue by category (Rs. crore)"); ax.set_xlabel("Revenue (Rs. cr)")
save(fig, "03_category_revenue")
share = (cat/cat.sum()*100).sort_values(ascending=False).round(2)
share.to_csv(f"{TABLES}/category_revenue_share.csv")
share

category
Electronics         57.41
Sports & Fitness    17.38
Home & Kitchen       8.54
Fashion              5.18
Automotive           4.32
Office Supplies      3.25
Toys & Baby          1.63
Beauty & Health      1.15
Grocery              0.65
Books                0.49
Name: total_price, dtype: float64

**Observation:** Electronics is the single largest revenue contributor thanks to high unit
prices, followed by Fashion and Home & Kitchen. No category exceeds 60% of revenue, so the
business is not dangerously concentrated, but the top three categories deserve first call on
marketing spend.

## Chart 4 - Order status distribution (donut)

In [6]:
st = orders["status"].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(st.values, labels=st.index, autopct="%1.1f%%", startangle=90,
       wedgeprops=dict(width=0.42), colors=sns.color_palette("Set2", len(st)))
ax.set_title("Order status distribution")
save(fig, "04_order_status_donut")
st.to_csv(f"{TABLES}/order_status.csv")
st

status
Delivered     32499
Cancelled      5996
Shipped        5071
Returned       3933
Processing     2501
Name: count, dtype: int64

**Observation:** Delivered orders dominate the mix, but cancellations and returns together
account for a meaningful slice of volume. Each cancelled order is revenue that was won and then
lost, so reducing this share is the cheapest growth available.

## Chart 5 - Top 10 states by number of orders

In [7]:
top_states = orders["state"].value_counts().head(10).sort_values()
fig, ax = plt.subplots()
ax.barh(top_states.index, top_states.values, color="#e76f51")
ax.set_title("Top 10 states by order count"); ax.set_xlabel("Orders")
save(fig, "05_top_states")
orders["state"].value_counts().head(10).to_csv(f"{TABLES}/top_states.csv")

**Observation:** Demand is led by the large urbanised states, and the top ten states carry the
majority of all orders. Warehouse placement and same-state delivery promises should follow this
ranking.

## Chart 6 - Customer segment distribution

In [8]:
seg = customers["segment"].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(seg.values, labels=seg.index, autopct="%1.1f%%", startangle=120,
       colors=sns.color_palette("pastel", len(seg)))
ax.set_title("Customer segment mix")
save(fig, "06_customer_segments")
seg.to_csv(f"{TABLES}/customer_segments.csv")
seg

segment
Regular     4026
Budget      2505
Premium     1501
New         1488
Inactive     480
Name: count, dtype: int64

**Observation:** The base is spread across the segments rather than being dominated by one
group. Premium customers are a minority of the base but, as Chart 10 shows, they punch above
their weight on order value.

## Chart 7 - Payment method usage

In [9]:
pm = orders["payment_method"].value_counts()
fig, ax = plt.subplots()
sns.barplot(x=pm.values, y=pm.index, ax=ax, palette="crest")
ax.set_title("Orders by payment method"); ax.set_xlabel("Orders")
save(fig, "07_payment_methods")
pm.to_csv(f"{TABLES}/payment_methods.csv")
pm

/tmp/ipykernel_2065/3857715149.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=pm.values, y=pm.index, ax=ax, palette="crest")


payment_method
UPI                 17570
Credit Card         10007
Debit Card           7624
Cash on Delivery     5883
Net Banking          4026
EMI                  2943
Wallet               1947
Name: count, dtype: int64

**Observation:** Digital methods (UPI and cards) carry most of the volume, but Cash on Delivery
remains material. COD is the mode most associated with cancellations and refusal at the door, so
nudging COD users towards prepaid is a direct lever on the cancellation rate.

## Chart 8 - Age distribution of customers

In [10]:
fig, ax = plt.subplots()
sns.histplot(customers["age"].dropna(), bins=30, color="#6a4c93", ax=ax)
ax.set_title("Customer age distribution"); ax.set_xlabel("Age"); ax.set_ylabel("Customers")
save(fig, "08_age_distribution")
bands = pd.cut(customers["age"], [17, 25, 35, 45, 55, 100],
               labels=["18-25", "26-35", "36-45", "46-55", "56+"]).value_counts().sort_index()
bands.to_csv(f"{TABLES}/age_bands.csv")
bands

age
18-25    1628
26-35    2074
36-45    2122
46-55    2026
56+      2150
Name: count, dtype: int64

**Observation:** Shoppers cluster in the working-age bands, with the 26-45 group the largest.
Creative and product assortment should be tuned to this group, while the 56+ tail is small enough
to be treated as a niche.

## Chart 9 - Return reasons

In [11]:
rr = returns["reason"].value_counts().sort_values()
fig, ax = plt.subplots()
ax.barh(rr.index, rr.values, color="#bc4749")
ax.set_title("Return reasons"); ax.set_xlabel("Returns")
save(fig, "09_return_reasons")
returns["reason"].value_counts().to_csv(f"{TABLES}/return_reasons.csv")
returns["reason"].value_counts()

reason
Size Issue                 1196
Changed Mind               1150
Damaged in Transit         1143
Wrong Product Delivered    1135
Defective Product          1134
Not as Described           1125
Duplicate Order            1081
Quality Issue              1075
Delayed Delivery            484
Better Price Found          477
Name: count, dtype: int64

**Observation:** Size and expectation-mismatch reasons lead the list, which points at catalogue
quality (sizing charts, photography, descriptions) rather than at logistics. Damage-in-transit
reasons are the second theme and belong with the courier partners.

## Chart 10 - Average Order Value by customer segment

In [12]:
d = oc[oc["status"] == "Delivered"]
aov = d.groupby("segment")["final_amount"].mean().sort_values(ascending=False)
fig, ax = plt.subplots()
sns.barplot(x=aov.index, y=aov.values, ax=ax, palette="flare")
ax.set_title("Average Order Value by segment (delivered orders)"); ax.set_ylabel("AOV (Rs.)")
for i, v in enumerate(aov.values):
    ax.text(i, v, inr(v), ha="center", va="bottom", fontsize=9)
save(fig, "10_aov_by_segment")
aov.round(0).to_csv(f"{TABLES}/aov_by_segment.csv")
aov.round(2)

/tmp/ipykernel_2065/3005701026.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=aov.index, y=aov.values, ax=ax, palette="flare")


segment
Budget      64951.14
Inactive    63638.69
Regular     63458.41
New         63277.66
Premium     61691.60
Name: final_amount, dtype: float64

**Observation:** Premium customers post the highest average order value, confirming that the
segment labels are meaningful. Moving even a small share of Regular customers into Premium
behaviour is worth more than acquiring the same number of Budget customers.

In [13]:
print("Charts saved:")
for f in sorted(os.listdir(CHARTS)):
    print(" -", f)

Charts saved:
 - 01_monthly_order_volume.png
 - 02_monthly_gmv.png
 - 02b_gmv_yoy.png
 - 03_category_revenue.png
 - 04_order_status_donut.png
 - 05_top_states.png
 - 06_customer_segments.png
 - 07_payment_methods.png
 - 08_age_distribution.png
 - 09_return_reasons.png
 - 10_aov_by_segment.png
